# Benchmark Extract KG - Relationship Extraction with DeepSeek API

Notebook nay su dung DeepSeek API de trich xuat quan he tu sach giao khoa lich su.

## Che do xu ly:
- **JSON Hierarchical** (mac dinh): Xu ly theo cau truc semantic (Chu de > Bai > Section > Subsection)
- **TXT Legacy**: Xu ly theo windows co dinh tu file text

### Cell 1: Nhap API Key (duoc che khi nhap)
import os
import sys
from getpass import getpass

# Them Extract_kg folder vao path
sys.path.insert(0, 'Extract_kg')

# Kiem tra API key tu environment truoc
api_key = os.environ.get('DEEPSEEK_API_KEY', '')

if not api_key:
    print('DEEPSEEK_API_KEY khong tim thay trong environment.')
    api_key = getpass('Nhap DeepSeek API Key (an khi nhap): ')
    
    if api_key:
        os.environ['DEEPSEEK_API_KEY'] = api_key
        print(f'API Key da duoc set: {api_key[:8]}...{api_key[-4:]}')
    else:
        print('ERROR: Khong co API key!')
else:
    print(f'API Key tu environment: {api_key[:8]}...{api_key[-4:]}')

print('Ready!')

In [2]:
# Cell 2: Import va kiem tra config
import config

print('=== CAU HINH ===')
print(f'USE_JSON_FORMAT: {getattr(config, "USE_JSON_FORMAT", False)}')
print(f'JSON_INPUT_FILE: {getattr(config, "JSON_INPUT_FILE", "N/A")}')
print(f'DEEPSEEK_MODEL: {getattr(config, "DEEPSEEK_MODEL", "deepseek-chat")}')
print(f'ENTITY_FILE: {config.ENTITY_FILE}')
print(f'OUTPUT_JSON: {config.OUTPUT_JSON}')

# Kiem tra JSON file
json_file = os.path.join(config.ROOT_DIR, getattr(config, 'JSON_INPUT_FILE', ''))
if os.path.exists(json_file):
    print(f'\n[OK] JSON file exists: {json_file}')
else:
    print(f'\n[ERROR] JSON file NOT found: {json_file}')

=== CAU HINH ===
USE_JSON_FORMAT: True
JSON_INPUT_FILE: SGK\SGK_Lich_Su_12_Ket_Noi_Tri_Thuc.json
DEEPSEEK_MODEL: deepseek-chat
ENTITY_FILE: D:\KLTN\KLTN\outputs\entities\entities_20251210_103252.json
OUTPUT_JSON: D:\KLTN\KLTN\outputs\kg\knowledge_graph_historical_v4.json

[OK] JSON file exists: D:\KLTN\KLTN\SGK\SGK_Lich_Su_12_Ket_Noi_Tri_Thuc.json


In [3]:
# Cell 3: Test DeepSeek API
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)

print('Testing DeepSeek API connection...')
response = client.chat.completions.create(
    model='deepseek-chat',
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant'},
        {'role': 'user', 'content': 'Reply OK if you can read this'}
    ],
    stream=False
)

print(f'Response: {response.choices[0].message.content}')
print('[OK] DeepSeek API working!')

Testing DeepSeek API connection...
Response: OK
[OK] DeepSeek API working!


In [4]:
# Cell 4: Load Entities
import json

# Load entities tu file
entity_file = config.ENTITY_FILE
print(f'Loading entities from: {entity_file}')

if os.path.exists(entity_file):
    with open(entity_file, 'r', encoding='utf-8') as f:
        entities = json.load(f)
    print(f'Loaded {len(entities)} entities')
    
    # Tao lookup
    entity_lookup = {e['id']: e for e in entities}
    
    # Thong ke theo type
    type_counts = {}
    for e in entities:
        t = e.get('type', 'Unknown')
        type_counts[t] = type_counts.get(t, 0) + 1
    
    print('\n=== THONG KE ENTITIES ===')
    for t, count in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f'  {t}: {count}')
else:
    print(f'[ERROR] Entity file not found: {entity_file}')
    entities = []
    entity_lookup = {}

Loading entities from: D:\KLTN\KLTN\outputs\entities\entities_20251210_103252.json
Loaded 625 entities

=== THONG KE ENTITIES ===
  Địa điểm: 147
  Tổ chức: 103
  Khái niệm: 86
  Văn kiện/Hiệp định: 59
  Sự kiện: 50
  Quốc gia: 44
  Chiến lược/Chủ trương: 36
  Hội nghị: 30
  Chiến dịch/Trận đánh: 27
  Nhân Vật: 25
  Công trình: 18


In [5]:
# Cell 5: Xem JSON stats (neu dung JSON mode)
if getattr(config, 'USE_JSON_FORMAT', False):
    try:
        from json_processor import JSONTextbookProcessor
        
        json_path = os.path.join(config.ROOT_DIR, config.JSON_INPUT_FILE)
        processor = JSONTextbookProcessor(json_path)
        stats = processor.get_statistics()
        
        print('=== THONG KE JSON SACH GIAO KHOA ===')
        print(f'Tong chunks (semantic units): {stats["total_chunks"]}')
        print(f'Tong chu de: {stats["total_topics"]}')
        print(f'Tong bai: {stats["total_lessons"]}')
        print(f'Tong cau: {stats["total_sentences"]}')
        print(f'Trung binh cau/chunk: {stats["avg_sentences_per_chunk"]:.1f}')
        
        print('\n[OK] json_processor ready!')
    except Exception as e:
        print(f'Error loading JSON: {e}')
else:
    print('TXT mode - skipping JSON stats')

=== THONG KE JSON SACH GIAO KHOA ===
Tong chunks (semantic units): 76
Tong chu de: 6
Tong bai: 17
Tong cau: 782
Trung binh cau/chunk: 10.3

[OK] json_processor ready!


In [6]:
# Cell 6: Extract relationships (JSON mode)
from relationship_processor import process_json_chunks_for_relationships

json_path = os.path.join(config.ROOT_DIR, config.JSON_INPUT_FILE)

print('=' * 60)
print('STARTING RELATIONSHIP EXTRACTION (JSON MODE)')
print('=' * 60)

# Trich xuat quan he (limit 10 chunks de test)
relationships = process_json_chunks_for_relationships(
    json_path=json_path,
    entity_lookup=entity_lookup,
    max_chunks=None  # Test voi 10 chunks dau tien
)

print(f'\nTotal relationships extracted: {len(relationships)}')

STARTING RELATIONSHIP EXTRACTION (JSON MODE)

[INFO] Đang chia chunks với overlap (7 câu/chunk, 5 câu overlap)...
   [Split] Chủ đề 1/Bài 1/Section 1/a: 8 sentences, 1856 chars -> 3 parts
   [Split] Chủ đề 1/Bài 1/Section 1/b: 9 sentences, 1557 chars -> 4 parts
   [Split] Chủ đề 1/Bài 1/Section 2/b: 7 sentences, 1526 chars -> 1 parts
   [Split] Chủ đề 1/Bài 2/Section 1/a: 13 sentences, 2085 chars -> 6 parts
   [Split] Chủ đề 1/Bài 2/Section 1/b: 18 sentences, 3020 chars -> 8 parts
   [Split] Chủ đề 1/Bài 2/Section 2/a: 7 sentences, 1268 chars -> 1 parts
   [Split] Chủ đề 1/Bài 3/Section 1/: 5 sentences, 1473 chars -> 1 parts
   [Split] Chủ đề 1/Bài 3/Section 2/b: 11 sentences, 1823 chars -> 5 parts
   [Split] Chủ đề 2/Bài 4/Section 1/a: 7 sentences, 1343 chars -> 1 parts
   [Split] Chủ đề 2/Bài 4/Section 2/b: 4 sentences, 1565 chars -> 1 parts
   [Split] Chủ đề 2/Bài 5/Section 3/: 10 sentences, 2094 chars -> 4 parts
   [Split] Chủ đề 3/Bài 6/Section 1/: 11 sentences, 1614 chars -> 5 pa

In [7]:
# Cell 7: Xem va luu ket qua
print('=== MAU RELATIONSHIPS ===')
for rel in relationships[:10]:
    print(f"\n{rel.get('subject_id')} --[{rel.get('predicate')}]--> {rel.get('object_id')}")
    print(f"   Evidence: {rel.get('evidence', '')[:100]}...")

# Luu ket qua
output_file = config.OUTPUT_JSON
os.makedirs(os.path.dirname(output_file), exist_ok=True)

kg_data = {
    'entities': entities,
    'relationships': relationships
}

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(kg_data, f, ensure_ascii=False, indent=2)

print(f'\n[OK] Saved to: {output_file}')

=== MAU RELATIONSHIPS ===

Liên Xô --[hop_tac]--> Mỹ
   Evidence: Trong bối cảnh đó, các nước Liên Xô, Mỹ, Anh triển khai các hoạt động để thành lập Liên hợp quốc....

Liên Xô --[tham_gia]--> Hội nghị Tê-hê-ran
   Evidence: Tại Hội nghị Tê-hê-ran (I-ran, từ ngày 28 – 11 đến ngày 1 – 12 – 1943), ba nước Liên Xô, Mỹ, Anh khẳ...

Liên Xô --[tham_gia]--> Hội nghị I-an-ta
   Evidence: Tại Hội nghị I-an-ta (Liên Xô, tháng 2 – 1945), ba nước Liên Xô, Mỹ, Anh đã ra quyết định về việc th...

Mỹ --[tham_gia]--> Hội nghị Tê-hê-ran
   Evidence: Tại Hội nghị Tê-hê-ran (I-ran, từ ngày 28 – 11 đến ngày 1 – 12 – 1943), ba nước Liên Xô, Mỹ, Anh khẳ...

Mỹ --[tham_gia]--> Hội nghị I-an-ta
   Evidence: Tại Hội nghị I-an-ta (Liên Xô, tháng 2 – 1945), ba nước Liên Xô, Mỹ, Anh đã ra quyết định về việc th...

Hội nghị Tê-hê-ran --[dien_ra_tai]--> I-ran
   Evidence: Tại Hội nghị Tê-hê-ran (I-ran, từ ngày 28 – 11 đến ngày 1 – 12 – 1943)......

Hội nghị I-an-ta --[dien_ra_tai]--> Liên Xô
   Evidence: Tại Hội ng

In [8]:
# Cell 8: Thong ke quan he
predicate_counts = {}
for rel in relationships:
    p = rel.get('predicate', 'Unknown')
    predicate_counts[p] = predicate_counts.get(p, 0) + 1

print('=== THONG KE QUAN HE ===')
for p, count in sorted(predicate_counts.items(), key=lambda x: -x[1])[:15]:
    print(f'  {p}: {count}')

=== THONG KE QUAN HE ===
  lien_quan_den: 520
  thuoc_ve: 124
  dien_ra_tai: 119
  liên_quan_đến: 116
  ket_qua_cua: 94
  hop_tac: 86
  chien_dau: 70
  tham_gia: 69
  thong_qua: 60
  lanh_dao: 54
  ky_ket: 37
  nguyen_nhan: 36
  diễn_ra_tại: 36
  thông_qua: 31
  thuộc_về: 31
